## Movement analysis (Study 1)

The following script: 
1. Pre-processes the movement data.
1. Calculates alpha and RMS for each participant.
2. Calculates complexity matching and linear covariance within actual pairs and pseudo pairs.
3. Calculates complexity matching and linear covariance between each participant and the background noise envelope (using both actual and pseudo pairings).
4. Merges the complexity matching datasets at each step.

### Calculates Alpha and RMS

In [1]:
# Import required libraries
import os
import pandas as pd
from IPython.display import clear_output
from utils.cleaning_utils import filter_data
from utils.dfa_utils import perform_dfa
from utils.feature_utils import calculate_distances, compute_rms

# ==============================
# Parameters
# ==============================
directory = "data/study1"                    # Path to input files
window_size = 940                            # ~20s at 47 fps
overlap = 0.5                                # 50% overlap
step_size = int(window_size * (1 - overlap)) # Step size
fps = 47                                     # Sampling rate (Hz)

# ==============================
# Main loop
# ==============================
csv_results = []
num_files = len([f for f in os.listdir(directory) if f.endswith('.csv')])
index = 0

for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        file_path = os.path.join(directory, filename)
        data = pd.read_csv(file_path)
        index += 1

        # Extract conditions from filename (new naming: PAIR_ENV_COND.csv)
        pair = filename.split('_')[0]
        env = filename.split('_')[1]
        cond = filename.split('_')[2].replace('.csv', '')

        # Apply Butterworth filter to raw data
        data = filter_data(data, cutoff=20, fs=fps, order=2)

        # Select relevant columns
        headVectP1 = data.iloc[:, 0:3].values  
        headVectP2 = data.iloc[:, 3:6].values

        # Calclate euclidean distances for both sets of coordinates
        data['headVectP1'] = calculate_distances(headVectP1)
        data['headVectP2'] = calculate_distances(headVectP2)

        # Select new columns to analyze
        columns = ['headVectP1', 'headVectP2']
        data = data[columns]

        # Remove the first row with NaN values
        data = data[1:].reset_index(drop=True) 

        # Windowed analysis
        for start in range(0, len(data) - window_size + 1, step_size):
            window = data[start:start + window_size]

            # === RMS (on raw, centred signals) ===
            rms_p1 = compute_rms(window['headVectP1'] - window['headVectP1'].mean())
            rms_p2 = compute_rms(window['headVectP2'] - window['headVectP2'].mean())

            # === DFA (on a deep copy so z-scoring doesn’t affect RMS) ===
            dfa_results = perform_dfa(window.copy(deep=True))
            dfa_p1 = dfa_results['headVectP1']
            dfa_p2 = dfa_results['headVectP2']

            # Append row
            csv_results.append({
                'pair': pair,
                'environment': env,
                'condition': cond,
                'window_index': start // step_size,
                'window_start': start,
                'window_end': start + window_size,
                'head_mag_rms_p1': rms_p1,
                'head_mag_rms_p2': rms_p2,
                'head_mag_alpha_p1': dfa_p1,
                'head_mag_alpha_p2': dfa_p2
            })

        clear_output(wait=True)
        print(f'Processed {index}/{num_files} files')

# ==============================
# Save results
# ==============================
output_path = f'results/study1/study1_dfa_rms.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
pd.DataFrame(csv_results).to_csv(output_path, index=False)

print(f'Results saved to {output_path}')

Processed 308/308 files
Results saved to results/study1/study1_dfa_rms.csv


### Actual Pairings

#### Calculates Dyadic Complexity Matching & RMS Cross-Correlation

In [2]:
# Import required libraries
import pandas as pd
import os
from utils.feature_utils import safe_corr, fisher_z

# ==============================
# Load window-level results
# ==============================
input_path = "results/study1/study1_dfa_rms.csv"
df = pd.read_csv(input_path)

# ==============================
# Group and correlate
# ==============================
corr_results = []

for (pair, environment, condition), group in df.groupby(["pair", "environment", "condition"]):
    # RMS correlation across windows
    rms_r = safe_corr(group["head_mag_rms_p1"], group["head_mag_rms_p2"])
    rms_z = fisher_z(rms_r)
    
    # DFA correlation across windows
    dfa_r = safe_corr(group["head_mag_alpha_p1"], group["head_mag_alpha_p2"])
    dfa_z = fisher_z(dfa_r)
    
    corr_results.append({
        "pair": pair,
        "environment": environment,
        "condition": condition,
        "n_windows": len(group),
        "rms_corr": rms_r,
        "rms_corr_z": rms_z,
        "alpha_corr": dfa_r,
        "alpha_corr_z": dfa_z
    })

# ==============================
# Save results
# ==============================
corr_df = pd.DataFrame(corr_results)
output_path = "results/study1/study1_dfa_rms_corr.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
corr_df.to_csv(output_path, index=False)

print(f"Correlation results saved → {output_path}")

Correlation results saved → results/study1/study1_dfa_rms_corr.csv


#### Calculates Environmental Complexity Matching and RMS Cross-Correlation

In [3]:
# Import required libraries
import os
import pandas as pd
import numpy as np
from utils.feature_utils import safe_corr, fisher_z

# ==============================
# Load stimulus information
# ==============================
noise_dir = "stimuli/study1"
noise_dfs = {}
for file in os.listdir(noise_dir):
    if file.endswith("_windows.csv"):
        env_name = "".join([part for part in file.replace("_windows.csv","").split(" ") if not part.isdigit()]).strip()
        df_noise = pd.read_csv(os.path.join(noise_dir, file))
        df_noise["environment"] = env_name
        noise_dfs[env_name] = df_noise

noise_all = pd.concat(noise_dfs.values(), ignore_index=True)

# Merge noise values into df
df = df.merge(
    noise_all,
    left_on=["environment", "window_index"],
    right_on=["environment", "window_index"],
    how="left"
)

# ==============================
# Compute participant-stimulus correlations
# ==============================
noise_results = []
for (pair, environment, condition), group in df.groupby(["pair", "environment", "condition"]):
    rms_corr_stimulus = np.nanmean([
        safe_corr(group["head_mag_rms_p1"], group["rms_noise"]),
        safe_corr(group["head_mag_rms_p2"], group["rms_noise"])
    ])
    alpha_corr_stimulus = np.nanmean([
        safe_corr(group["head_mag_alpha_p1"], group["alpha_noise"]),
        safe_corr(group["head_mag_alpha_p2"], group["alpha_noise"])
    ])

    noise_results.append({
        "pair": pair,
        "environment": environment,
        "condition": condition,
        "n_windows": len(group),
        "rms_corr_stimulus": rms_corr_stimulus,
        "rms_corr_z_stimulus": fisher_z(rms_corr_stimulus),
        "alpha_corr_stimulus": alpha_corr_stimulus,
        "alpha_corr_z_stimulus": fisher_z(alpha_corr_stimulus)
    })

noise_df = pd.DataFrame(noise_results)

# ==============================
# Merge noise correlations into partner correlations
# ==============================
corr_df = corr_df.merge(
    noise_df,
    on=["pair", "environment", "condition", "n_windows"],
    how="left"
)

# ==============================
# Save merged file
# ==============================
output_path = "results/study1/study1_dfa_rms_corr_with_noise.csv"
corr_df.to_csv(output_path, index=False)

print(f"Partner + noise correlations saved → {output_path}")

Partner + noise correlations saved → results/study1/study1_dfa_rms_corr_with_noise.csv


### Pseudo Pairings

#### Calculates Pseudo Complexity Matching & RMS Cross-Correlation

In [4]:
# Import required libraries
import pandas as pd
import numpy as np
import os
from utils.feature_utils import safe_corr, fisher_z

# ==============================
# Load window-level results
# ==============================
input_path = "results/study1/study1_dfa_rms.csv"
df = pd.read_csv(input_path)

# ==============================
# Prepare long-format data
# ==============================
df_long = pd.DataFrame({
    "pair": np.repeat(df["pair"].values, 2),
    "environment": np.repeat(df["environment"].values, 2),
    "condition": np.repeat(df["condition"].values, 2),
    "window_start": np.repeat(df["window_start"].values, 2),
    "window_end": np.repeat(df["window_end"].values, 2),
    "role": ["p1", "p2"] * len(df),
    "head_mag_rms": pd.concat([df["head_mag_rms_p1"], df["head_mag_rms_p2"]]).values,
    "head_mag_alpha": pd.concat([df["head_mag_alpha_p1"], df["head_mag_alpha_p2"]]).values
})

# # ==============================
# Pseudo-pairs analysis (pooled across roles, Fisher z before averaging)
# ==============================
pseudo_results = []

for (environment, condition), group in df_long.groupby(["environment", "condition"]):
    # Iterate through each true pair
    for pair_id in group["pair"].unique():
        # extract the two real participants
        p1 = group[(group["pair"] == pair_id) & (group["role"] == "p1")]
        p2 = group[(group["pair"] == pair_id) & (group["role"] == "p2")]

        if p1.empty or p2.empty:
            continue

        # Collect all pseudo correlations for this dyad
        pseudo_rms_corrs, pseudo_alpha_corrs = [], []

        for other_pair in group["pair"].unique():
            if other_pair == pair_id:
                continue

            for role in ["p1", "p2"]:
                other = group[(group["pair"] == other_pair) & (group["role"] == role)]
                if other.empty:
                    continue

                # align on windows
                merged_p1 = pd.merge(p1, other, on=["window_start", "window_end"], suffixes=("", "_other"))
                merged_p2 = pd.merge(p2, other, on=["window_start", "window_end"], suffixes=("", "_other"))

                # compute correlations
                for merged in [merged_p1, merged_p2]:
                    rms_r = safe_corr(merged["head_mag_rms"], merged["head_mag_rms_other"])
                    alpha_r = safe_corr(merged["head_mag_alpha"], merged["head_mag_alpha_other"])

                    if not pd.isna(rms_r):
                        pseudo_rms_corrs.append(fisher_z(rms_r))   # z-transform *before* averaging
                    if not pd.isna(alpha_r):
                        pseudo_alpha_corrs.append(fisher_z(alpha_r))

        # average Fisher z values, then back-transform
        mean_rms_z = np.nanmean(pseudo_rms_corrs) if pseudo_rms_corrs else np.nan
        mean_alpha_z = np.nanmean(pseudo_alpha_corrs) if pseudo_alpha_corrs else np.nan

        pseudo_results.append({
            "pair": pair_id,
            "environment": environment,
            "condition": condition,
            "n_windows": len(group["window_start"].unique()),
            "rms_pseudo_corr": np.tanh(mean_rms_z) if not pd.isna(mean_rms_z) else np.nan,
            "rms_pseudo_corr_z": mean_rms_z,
            "alpha_pseudo_corr": np.tanh(mean_alpha_z) if not pd.isna(mean_alpha_z) else np.nan,
            "alpha_pseudo_corr_z": mean_alpha_z
        })

# ==============================
# Save results
# ==============================
pseudo_df = pd.DataFrame(pseudo_results)
output_path = "results/study1/study1_dfa_rms_pseudo_corr.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
pseudo_df.to_csv(output_path, index=False)

print(f"Pseudo correlations saved → {output_path}")

Pseudo correlations saved → results/study1/study1_dfa_rms_pseudo_corr.csv


#### Calculates Environmental Complexity Matching and RMS Cross-Correlation

In [5]:
# ==============================
# Pseudo participant–stimulus correlations
# ==============================
pseudo_stimulus_results = []

# loop over env × condition
for (environment, condition), group in df.groupby(["environment", "condition"]):
    # iterate through each real pair
    for pair_id in group["pair"].unique():
        p1 = group[(group["pair"] == pair_id)][["window_index", "head_mag_rms_p1", "head_mag_alpha_p1"]]
        p2 = group[(group["pair"] == pair_id)][["window_index", "head_mag_rms_p2", "head_mag_alpha_p2"]]

        if p1.empty or p2.empty:
            continue

        pseudo_rms_z, pseudo_alpha_z = [], []

        # loop over *other* environments (pseudo stimuli)
        for other_env in df["environment"].unique():
            if other_env == environment:
                continue

            # grab the noise windows for the other environment
            noise_other = noise_all[noise_all["environment"] == other_env][
                ["window_index", "rms_noise", "alpha_noise"]
            ]

            # align participants with wrong stimulus by window index
            merged_p1 = pd.merge(p1, noise_other, on="window_index", how="inner")
            merged_p2 = pd.merge(p2, noise_other, on="window_index", how="inner")

            # compute correlations + Fisher z
            for merged, rms_col, alpha_col in [
                (merged_p1, "head_mag_rms_p1", "head_mag_alpha_p1"),
                (merged_p2, "head_mag_rms_p2", "head_mag_alpha_p2"),
            ]:
                rms_r = safe_corr(merged[rms_col], merged["rms_noise"])
                alpha_r = safe_corr(merged[alpha_col], merged["alpha_noise"])
                if not pd.isna(rms_r):
                    pseudo_rms_z.append(fisher_z(rms_r))
                if not pd.isna(alpha_r):
                    pseudo_alpha_z.append(fisher_z(alpha_r))

        # average Fisher z’s, back-transform
        mean_rms_z = np.nanmean(pseudo_rms_z) if pseudo_rms_z else np.nan
        mean_alpha_z = np.nanmean(pseudo_alpha_z) if pseudo_alpha_z else np.nan

        pseudo_stimulus_results.append({
            "pair": pair_id,
            "environment": environment,
            "condition": condition,
            "n_windows": len(group["window_index"].unique()),
            "rms_pseudo_corr_stimulus": np.tanh(mean_rms_z) if not pd.isna(mean_rms_z) else np.nan,
            "rms_pseudo_corr_z_stimulus": mean_rms_z,
            "alpha_pseudo_corr_stimulus": np.tanh(mean_alpha_z) if not pd.isna(mean_alpha_z) else np.nan,
            "alpha_pseudo_corr_z_stimulus": mean_alpha_z
        })

# convert to DataFrame
pseudo_stimulus_df = pd.DataFrame(pseudo_stimulus_results)

# ==============================
# Merge into pseudo_df (partner pseudo)
# ==============================
pseudo_df = pseudo_df.merge(
    pseudo_stimulus_df,
    on=["pair", "environment", "condition", "n_windows"],
    how="left"
)

# save merged file
output_path = "results/study1/study1_dfa_rms_pseudo_corr_with_noise.csv"
pseudo_df.to_csv(output_path, index=False)
print(f"Pseudo correlations (partner + stimulus) saved → {output_path}")

Pseudo correlations (partner + stimulus) saved → results/study1/study1_dfa_rms_pseudo_corr_with_noise.csv
